# 使用 ColabFold 进行蛋白质结构预测

本教程使用 Jupyter notebook 演示如何使用 `ColabFold` 完成蛋白质结构预测。

> 适用场景：Linux / 服务器 / 本地 GPU 工作站。

## 本 notebook 的目标

1. 安装并检查 ColabFold 运行环境
2. 准备蛋白序列（FASTA）
3. 运行 `colabfold_batch` 进行结构预测
4. 读取并解释关键输出（pLDDT、PAE、rank）
5. 在 notebook 中可视化预测结构

## 1 说明与依赖

### 你将用到的核心工具

- `colabfold_batch`：ColabFold 命令行主程序
- `MMseqs2`：用于快速同源搜索与 MSA 构建
- `JAX` + `AlphaFold` 参数：完成结构推理

### 重要说明

- 建议在 Linux/WSL2 环境执行预测，Windows环境下可能会报错。
- 建议使用 GPU 运行代码，如果使用 CPU 跑通流程，则速度会较慢。

In [1]:
# 1) 环境检查
# 作用：确认 Python 版本、当前工作目录、GPU 可见性，便于排查运行问题。

import os
import sys
import platform
import subprocess
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
print("CWD:", os.getcwd())

# 检查 NVIDIA GPU
try:
    out = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    if out.returncode == 0:
        print("\n[nvidia-smi 输出]\n")
        print(out.stdout[:1200])  # 仅展示前 1200 字符
    else:
        print("未检测到可用 nvidia-smi（可能没有 NVIDIA GPU 或未安装驱动）。")
except FileNotFoundError:
    print("系统中未找到 nvidia-smi 命令。")

Python: 3.11.1 (main, Feb  6 2026, 08:59:00) [GCC 13.3.0]
Platform: Linux-6.6.92-34.1.tl4.x86_64-x86_64-with-glibc2.39
CWD: /

[nvidia-smi 输出]

Fri Mar 27 13:05:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.65.06              Driver Version: 580.65.06      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       On  |   00000000:00:09.0 Off |                    0 |
| N/A   29C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Defaul

## 2 安装 ColabFold

下面给出两种常见安装方式：

- **方式 A（推荐）**：在终端里提前创建好环境，然后在 notebook 里直接调用。
- **方式 B**：直接在 notebook 中执行安装命令（更直观，但首次安装可能较慢）。

存在的 GPU 修复逻辑：

- 自动检测是否存在 NVIDIA GPU（`nvidia-smi`）
- 自动检测 JAX backend（`gpu/cpu`）
- 若检测到 GPU 但 JAX 仍为 CPU，则自动安装 CUDA 版 JAX

> 如果你已经安装好 ColabFold 且 JAX backend 已是 `gpu`，可以跳过下一格。

In [13]:
# 2) 环境检查 + 按需安装 ColabFold（含 JAX GPU 自动修复）
# 作用：先检查依赖，只有缺包或需要修复时才安装，避免重复安装。

import sys
import subprocess
import importlib.util

INSTALL_IF_MISSING = True        # True: 缺依赖时自动安装；False: 只检查不安装
AUTO_FIX_JAX_GPU = True          # 自动检测并修复 JAX GPU 后端
CUDA_TAG = "cuda12"             # 常见: cuda12；老环境可能需 cuda11


def run_cmd(cmd):
    print("执行命令:", " ".join(cmd))
    r = subprocess.run(cmd, check=False)
    if r.returncode != 0:
        raise RuntimeError(f"命令执行失败: {' '.join(cmd)}")


def detect_gpu():
    try:
        smi = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, check=False)
        return smi.returncode == 0 and ("GPU" in smi.stdout)
    except FileNotFoundError:
        return False


def jax_backend():
    try:
        import jax
        return jax.default_backend()
    except Exception:
        return "unknown"


# ---- 1) 依赖检查 ----
required_modules = ["colabfold", "alphafold", "Bio", "py3Dmol", "matplotlib", "pandas"]
missing_modules = [m for m in required_modules if importlib.util.find_spec(m) is None]

print("缺失依赖:", missing_modules if missing_modules else "无")

need_install = len(missing_modules) > 0

# ---- 2) 按需安装 ----
if need_install:
    if INSTALL_IF_MISSING:
        run_cmd([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
        run_cmd([sys.executable, "-m", "pip", "install", "-U", "colabfold[alphafold]", "biopython", "py3Dmol", "matplotlib", "pandas"])
        print("\n缺失依赖安装完成。")
    else:
        print("检测到缺失依赖，但 INSTALL_IF_MISSING=False，未执行安装。")
else:
    print("依赖已齐全，跳过安装步骤。")

# ---- 3) GPU + JAX 后端检查与按需修复 ----
has_gpu = detect_gpu()
backend = jax_backend()
print(f"NVIDIA GPU 检测: {has_gpu}")
print(f"JAX backend: {backend}")

if AUTO_FIX_JAX_GPU and has_gpu and backend != "gpu":
    print("\n检测到 GPU 存在但 JAX 非 GPU 后端，开始修复 JAX CUDA 后端...")
    run_cmd([
        sys.executable, "-m", "pip", "install", "-U",
        f"jax[{CUDA_TAG}]",
        "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
    ])
    print("JAX CUDA 后端安装完成，请重启 kernel 后再检查。")
elif has_gpu and backend == "gpu":
    print("JAX 已在 GPU 后端运行，无需修复。")
elif not has_gpu:
    print("未检测到 NVIDIA GPU，将保持 CPU 后端。")

缺失依赖: 无
依赖已齐全，跳过安装步骤。
NVIDIA GPU 检测: True
JAX backend: gpu
JAX 已在 GPU 后端运行，无需修复。


In [2]:
# 3) 检查 colabfold_batch 与 alphafold 依赖是否可用
# 作用：提前发现环境问题，避免在正式预测时报错中断。

import shutil
import subprocess
import importlib.util

exe = shutil.which("colabfold_batch")
print("colabfold_batch 路径:", exe)

af_installed = importlib.util.find_spec("alphafold") is not None
print("alphafold 模块是否可导入:", af_installed)

if exe is None:
    print("未找到 colabfold_batch。请先安装 ColabFold，或确认当前 Python 环境正确。")
else:
    out = subprocess.run([exe, "--help"], capture_output=True, text=True, check=False)
    print("\ncolabfold_batch --help（前 40 行）:\n")
    print("\n".join(out.stdout.splitlines()[:40]))

if not af_installed:
    print("\n检测到缺少 alphafold，请执行：")
    print(f"{sys.executable} -m pip install -U 'colabfold[alphafold]'")

colabfold_batch 路径: /root/.pyenv/versions/3.11.1/bin/colabfold_batch
alphafold 模块是否可导入: True

colabfold_batch --help（前 40 行）:

usage: colabfold_batch [-h] [--msa-only]
                       [--msa-mode {mmseqs2_uniref_env,mmseqs2_uniref_env_envpair,mmseqs2_uniref,single_sequence}]
                       [--pair-mode {unpaired,paired,unpaired_paired}]
                       [--pair-strategy {complete,greedy}] [--templates]
                       [--custom-template-path CUSTOM_TEMPLATE_PATH]
                       [--custom-template-cache-path CUSTOM_TEMPLATE_CACHE_PATH]
                       [--max-template-date MAX_TEMPLATE_DATE]
                       [--max-template-hits MAX_TEMPLATE_HITS]
                       [--pdb-hit-file PDB_HIT_FILE]
                       [--local-pdb-path LOCAL_PDB_PATH]
                       [--num-recycle NUM_RECYCLE]
                       [--recycle-early-stop-tolerance RECYCLE_EARLY_STOP_TOLERANCE]
                       [--num-ensemble NUM_ENSEMBLE

## 3 准备输入序列（FASTA）

这一部分用于创建输入的 FASTA 文件。

- 你可以替换为自己的蛋白质序列。
- `job_name` 会用在输出目录命名中，便于管理多次实验。

In [9]:
# 4) 写入 FASTA 输入文件
# 作用：把蛋白序列保存成 colabfold_batch 可读取的标准 FASTA 格式。

from pathlib import Path

workspace_root = Path("/workspace")
workspace_root.mkdir(parents=True, exist_ok=True)
work_dir = (workspace_root / "colabfold_jobs").resolve()
work_dir.mkdir(parents=True, exist_ok=True)

job_name = "demo_protein"
sequence = (
    "MSTNPKPQRKTKRNTNRRPQDVKFPGGGQIVGGVSKVSSS"
    "RRGPRLGVRGQGQGQGQGQGQGQGQGQGQGQGQGQGQ"  # 示例序列，可替换
)

fasta_path = work_dir / f"{job_name}.fasta"
with open(fasta_path, "w", encoding="utf-8") as f:
    f.write(f">{job_name}\n")
    f.write(sequence + "\n")

print("FASTA 文件已创建:", fasta_path)
print("序列长度:", len(sequence))

FASTA 文件已创建: /workspace/colabfold_jobs/demo_protein.fasta
序列长度: 77


## 4 运行 ColabFold 预测（使用GPU）

下面的代码调用 `colabfold_batch`，并在运行前检查 GPU/JAX 后端。

- `--num-recycle`：循环精修次数，越高可能更准，但更慢。
- `--num-models`：使用的模型数。
- `--model-type`：模型类型，常见 `auto`。
- `--amber`：是否使用 Amber 进一步弛豫（提升几何质量，但更耗时）。
- `--use-gpu-relax`：若支持，使用 GPU 进行 relax 加速。

> 说明：如果检测不到 GPU，默认会直接报错停止（不再回退到 CPU）。

In [10]:
# 5) 执行结构预测（GPU 优先）
# 作用：运行 colabfold_batch，生成 PDB、JSON、日志与排序结果。

import os
import shlex
import subprocess

output_dir = work_dir / f"{job_name}_result"
output_dir.mkdir(parents=True, exist_ok=True)

run_prediction = True      # 准备好后改为 True
require_gpu = True          # True: 必须使用 GPU；False: 允许回退 CPU
use_gpu_relax = True        # 有 GPU 时启用 --use-gpu-relax

# ---- GPU 检查（nvidia-smi + jax）----
has_nvidia_gpu = False
try:
    smi = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, check=False)
    has_nvidia_gpu = (smi.returncode == 0 and ("GPU" in smi.stdout))
except FileNotFoundError:
    has_nvidia_gpu = False

jax_backend = "unknown"
try:
    import jax
    jax_backend = jax.default_backend()
except Exception:
    pass

print(f"检测到 NVIDIA GPU: {has_nvidia_gpu}")
print(f"JAX backend: {jax_backend}")

if require_gpu and (not has_nvidia_gpu or jax_backend != "gpu"):
    raise RuntimeError(
        "当前环境未正确启用 GPU。\n"
        "请先安装 CUDA 版 JAX，并确认 nvidia-smi 可用后再运行。\n"
        "可参考：pip install -U 'jax[cuda12]' -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html"
    )

cmd = [
    "colabfold_batch",
    "--num-recycle", "3",
    "--num-models", "5",
    "--model-type", "auto",
]

if use_gpu_relax:
    cmd.append("--use-gpu-relax")

cmd.extend([
    str(fasta_path),
    str(output_dir),
])

print("将执行命令:\n", " ".join(shlex.quote(x) for x in cmd))

if run_prediction:
    env = os.environ.copy()
    # 显式指定可见 GPU；如需指定单卡可改成具体编号（例如 "0"）
    env.setdefault("CUDA_VISIBLE_DEVICES", "0")
    proc = subprocess.run(cmd, check=False, env=env)
    if proc.returncode != 0:
        raise RuntimeError("colabfold_batch 执行失败，请检查环境和日志输出。")
    print("预测完成，结果目录:", output_dir)
else:
    print("已跳过预测（run_prediction=False）。")

检测到 NVIDIA GPU: True
JAX backend: gpu
将执行命令:
 colabfold_batch --num-recycle 3 --num-models 5 --model-type auto --use-gpu-relax /workspace/colabfold_jobs/demo_protein.fasta /workspace/colabfold_jobs/demo_protein_result
2026-03-27 13:23:25,546 Running colabfold 1.6.1



limited shared resource only capable of processing a few thousand MSAs per day. Please
submit jobs only from a single IP address. We reserve the right to limit access to the
server case-by-case when usage exceeds fair use. If you require more MSAs: You can 
precompute all MSAs with `colabfold_search` or host your own API and pass it to `--host-url`

I0000 00:00:1774617806.194175   30597 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774617807.836240   30597 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


2026-03-27 13:23:29,540 Running on GPU
2026-03-27 13:23:29,627 Found 5 citations for tools or databases
2026-03-27 13:23:29,627 Query 1/1: demo_protein (length 77)


COMPLETE: 100%|██████████| 150/150 [elapsed: 00:01 remaining: 00:00]


2026-03-27 13:23:31,211 Setting max_seq=512, max_extra_seq=561
2026-03-27 13:24:04,136 alphafold2_ptm_model_1_seed_000 recycle=0 pLDDT=61 pTM=0.129
2026-03-27 13:24:25,255 alphafold2_ptm_model_1_seed_000 recycle=1 pLDDT=61.7 pTM=0.13 tol=10.6
2026-03-27 13:24:29,952 alphafold2_ptm_model_1_seed_000 recycle=2 pLDDT=61 pTM=0.13 tol=3.55
2026-03-27 13:24:34,671 alphafold2_ptm_model_1_seed_000 recycle=3 pLDDT=60.9 pTM=0.131 tol=3.32
2026-03-27 13:24:34,672 alphafold2_ptm_model_1_seed_000 took 55.2s (3 recycles)
2026-03-27 13:24:39,432 alphafold2_ptm_model_2_seed_000 recycle=0 pLDDT=59.3 pTM=0.119
2026-03-27 13:24:44,178 alphafold2_ptm_model_2_seed_000 recycle=1 pLDDT=59.9 pTM=0.124 tol=8.8
2026-03-27 13:24:48,932 alphafold2_ptm_model_2_seed_000 recycle=2 pLDDT=60.2 pTM=0.128 tol=3.31
2026-03-27 13:24:53,711 alphafold2_ptm_model_2_seed_000 recycle=3 pLDDT=60.1 pTM=0.13 tol=1.24
2026-03-27 13:24:53,712 alphafold2_ptm_model_2_seed_000 took 19.0s (3 recycles)
2026-03-27 13:24:58,509 alphafold2_

## 5 结果文件解析

ColabFold 输出中常见内容：

- `*rank*.pdb`：按综合评分排序后的结构
- `*scores*.json` 或 `*result*.json`：含 pLDDT、PAE、ptm/ipTM 等信息
- 日志与中间文件：用于复现和排错

下面代码会自动扫描并汇总主要结果。

In [11]:
# 6) 汇总结果文件并提取关键指标
# 作用：按文件后缀判断 PDB（suffix == .pdb），兼容不同命名风格。

import json
import pandas as pd

if not output_dir.exists():
    print("结果目录不存在，请先运行预测。")
else:
    all_files = sorted(output_dir.glob("*"))
    all_pdb = [p for p in all_files if p.is_file() and p.suffix.lower() == ".pdb"]
    json_files = [p for p in all_files if p.is_file() and p.suffix.lower() == ".json"]

    print("PDB 数量(按后缀):", len(all_pdb))
    for p in all_pdb[:10]:
        print(" -", p.name)

    print("\nJSON 数量(按后缀):", len(json_files))
    for j in json_files[:10]:
        print(" -", j.name)

    rows = []
    for jf in json_files:
        try:
            data = json.loads(jf.read_text(encoding="utf-8"))
            row = {"file": jf.name}
            for k in ["ptm", "iptm", "ranking_confidence"]:
                if k in data and isinstance(data[k], (int, float)):
                    row[k] = float(data[k])
            rows.append(row)
        except Exception:
            continue

    if rows:
        df_score = pd.DataFrame(rows)
        sort_cols = [c for c in ["ranking_confidence", "ptm", "iptm"] if c in df_score.columns]
        if sort_cols:
            df_score = df_score.sort_values(by=sort_cols, ascending=False)
        display(df_score.head(20))
    else:
        print("未在 JSON 中读取到可汇总评分字段（不同版本输出字段可能不同）。")

PDB 数量(按后缀): 5
 - demo_protein_unrelaxed_rank_001_alphafold2_ptm_model_4_seed_000.pdb
 - demo_protein_unrelaxed_rank_002_alphafold2_ptm_model_1_seed_000.pdb
 - demo_protein_unrelaxed_rank_003_alphafold2_ptm_model_2_seed_000.pdb
 - demo_protein_unrelaxed_rank_004_alphafold2_ptm_model_3_seed_000.pdb
 - demo_protein_unrelaxed_rank_005_alphafold2_ptm_model_5_seed_000.pdb

JSON 数量(按后缀): 7
 - config.json
 - demo_protein_predicted_aligned_error_v1.json
 - demo_protein_scores_rank_001_alphafold2_ptm_model_4_seed_000.json
 - demo_protein_scores_rank_002_alphafold2_ptm_model_1_seed_000.json
 - demo_protein_scores_rank_003_alphafold2_ptm_model_2_seed_000.json
 - demo_protein_scores_rank_004_alphafold2_ptm_model_3_seed_000.json
 - demo_protein_scores_rank_005_alphafold2_ptm_model_5_seed_000.json


,file,ptm
2,demo_protein_scores_rank_001_alphafold2_ptm_mo...,0.13
3,demo_protein_scores_rank_002_alphafold2_ptm_mo...,0.13
4,demo_protein_scores_rank_003_alphafold2_ptm_mo...,0.13
5,demo_protein_scores_rank_004_alphafold2_ptm_mo...,0.12
6,demo_protein_scores_rank_005_alphafold2_ptm_mo...,0.12
0,config.json,NaN
1,demo_protein_predicted_aligned_error_v1.json,NaN


## 6 三维可视化最佳结构

这里使用 `py3Dmol` 在 notebook 内直接显示 PDB 结构。

- 颜色模式使用 `spectrum`（按残基顺序上色）
- 你也可以改成按 pLDDT 上色（需要进一步读取 B-factor）

In [12]:
# 7) 在 notebook 中可视化 .pdb 文件
# 作用：仅按后缀 .pdb 筛选并展示（不使用任何优先级规则）。

import py3Dmol

if not output_dir.exists():
    print("结果目录不存在，请先运行预测。")
else:
    pdb_files = [p for p in sorted(output_dir.glob("*")) if p.is_file() and p.suffix.lower() == ".pdb"]

    if not pdb_files:
        print("未找到任何 .pdb 文件。请确认预测已成功完成。")
    else:
        print("检测到以下 .pdb 文件：")
        for p in pdb_files:
            print(" -", p.name)

        # 直接使用后缀筛选后的第一个文件进行可视化
        target = pdb_files[0]
        pdb_str = target.read_text(encoding="utf-8", errors="ignore")
        view = py3Dmol.view(width=900, height=650)
        view.addModel(pdb_str, "pdb")
        view.setStyle({"cartoon": {"color": "spectrum"}})
        view.zoomTo()
        view.show()
        print("当前显示:", target.name)

检测到以下 .pdb 文件：
 - demo_protein_unrelaxed_rank_001_alphafold2_ptm_model_4_seed_000.pdb
 - demo_protein_unrelaxed_rank_002_alphafold2_ptm_model_1_seed_000.pdb
 - demo_protein_unrelaxed_rank_003_alphafold2_ptm_model_2_seed_000.pdb
 - demo_protein_unrelaxed_rank_004_alphafold2_ptm_model_3_seed_000.pdb
 - demo_protein_unrelaxed_rank_005_alphafold2_ptm_model_5_seed_000.pdb


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

当前显示: demo_protein_unrelaxed_rank_001_alphafold2_ptm_model_4_seed_000.pdb


## 7 如何解读结果

- **pLDDT（0-100）**：局部可信度，通常 >70 表示较可靠，>90 表示高可信。
- **PAE**：不同残基对之间的相对位置误差估计，越低越好。
- **ptm / ipTM**：整体拓扑与多聚体界面质量指标（多链场景尤为重要）。
- **ranked_0**：通常是综合评分最优模型，可作为主结果进一步分析。

In [8]:
# 8) 导出结果目录清单（可选）
# 作用：便于整理实验记录，确认输出文件完整性。

if output_dir.exists():
    files = sorted(output_dir.glob("*"))
    print(f"结果目录: {output_dir}")
    print(f"文件数量: {len(files)}")
    for f in files[:50]:
        print("-", f.name)
else:
    print("结果目录不存在，请先运行预测。")

结果目录: /colabfold_jobs/demo_protein_result
文件数量: 20
- cite.bibtex
- config.json
- demo_protein.a3m
- demo_protein.done.txt
- demo_protein_coverage.png
- demo_protein_env
- demo_protein_pae.png
- demo_protein_plddt.png
- demo_protein_predicted_aligned_error_v1.json
- demo_protein_scores_rank_001_alphafold2_ptm_model_1_seed_000.json
- demo_protein_scores_rank_002_alphafold2_ptm_model_4_seed_000.json
- demo_protein_scores_rank_003_alphafold2_ptm_model_2_seed_000.json
- demo_protein_scores_rank_004_alphafold2_ptm_model_3_seed_000.json
- demo_protein_scores_rank_005_alphafold2_ptm_model_5_seed_000.json
- demo_protein_unrelaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb
- demo_protein_unrelaxed_rank_002_alphafold2_ptm_model_4_seed_000.pdb
- demo_protein_unrelaxed_rank_003_alphafold2_ptm_model_2_seed_000.pdb
- demo_protein_unrelaxed_rank_004_alphafold2_ptm_model_3_seed_000.pdb
- demo_protein_unrelaxed_rank_005_alphafold2_ptm_model_5_seed_000.pdb
- log.txt
